In [1]:
import sys
sys.path.append("..")  # so we can import from src/

from src.data.loader import load_raw

df = load_raw("../data/raw/creditcard.csv")
print(df.shape)
print(df.dtypes.value_counts())

(284807, 31)
float32    29
int32       1
int8        1
Name: count, dtype: int64


In [2]:
mb = df.memory_usage(deep=True).sum() / (1024 ** 2)
print(f"Memory usage: {mb:.1f} MB")
print(f"Fraud rate: {df['Class'].mean() * 100:.3f}%")

Memory usage: 32.9 MB
Fraud rate: 0.173%


In [3]:
from src.data.loader import save_parquet, load_parquet

# Save once
save_parquet(df, "../data/processed/transactions.parquet")
print("Saved.")

# Verify by loading back
df2 = load_parquet("../data/processed/transactions.parquet")
print(df2.shape, df2.dtypes.value_counts().to_dict())

Saved.
(284807, 31) {dtype('float32'): 29, dtype('int32'): 1, dtype('int8'): 1}


In [4]:
import time

t0 = time.time()
_ = load_raw("../data/raw/creditcard.csv")
t_csv = time.time() - t0

t0 = time.time()
_ = load_parquet("../data/processed/transactions.parquet")
t_parquet = time.time() - t0

print(f"CSV load:     {t_csv:.2f} s")
print(f"Parquet load: {t_parquet:.2f} s")
print(f"Speedup:      {t_csv / t_parquet:.1f}x")

CSV load:     0.98 s
Parquet load: 0.04 s
Speedup:      25.3x


In [5]:
from src.data.splitter import time_based_split

train_df, val_df, test_df = time_based_split(df)

print(f"Train: {len(train_df):>7,} rows ({len(train_df)/len(df)*100:.1f}%)")
print(f"Val:   {len(val_df):>7,} rows ({len(val_df)/len(df)*100:.1f}%)")
print(f"Test:  {len(test_df):>7,} rows ({len(test_df)/len(df)*100:.1f}%)")

Train: 199,364 rows (70.0%)
Val:    42,721 rows (15.0%)
Test:   42,722 rows (15.0%)


In [6]:
for name, split in [("Train", train_df), ("Val", val_df), ("Test", test_df)]:
    n_fraud = split["Class"].sum()
    rate = split["Class"].mean() * 100
    print(f"{name:5s}: {n_fraud:4d} frauds / {len(split):>7,} rows ({rate:.3f}%)")

Train:  384 frauds / 199,364 rows (0.193%)
Val  :   56 frauds /  42,721 rows (0.131%)
Test :   52 frauds /  42,722 rows (0.122%)


In [7]:
from src.data.loader import save_parquet

save_parquet(train_df, "../data/processed/train.parquet")
save_parquet(val_df, "../data/processed/val.parquet")
save_parquet(test_df, "../data/processed/test.parquet")
print("Splits saved.")

Splits saved.
